# Seal OCR: rebuild v1 checkpoint, then fine-tune

Attach only `seal-recognition-v4-manual`. It already contains all 14,500 v3 samples plus the new manual samples. This notebook reconstructs the v3 splits from its manifest, rebuilds a resumable v1 checkpoint on two GPUs, then fine-tunes it on v4 at a lower learning rate.


In [ ]:
from pathlib import Path
from zipfile import ZipFile
import csv, hashlib, json, os, random, shutil, subprocess, sys, urllib.request

RUN_MODE = 'smoke'  # smoke: 1+1 epochs; full: 60+8 epochs
GPU_IDS = '0,1'
CONTROL_SEED = 20260919
FINETUNE_SEED = 20260921
SPACING_PROB = 0.35
MANUAL_REPEAT = 3
CONTROL_EPOCHS = 1 if RUN_MODE == 'smoke' else 60
FINETUNE_EPOCHS = 1 if RUN_MODE == 'smoke' else 8
if RUN_MODE not in {'smoke', 'full'}: raise ValueError(RUN_MODE)
os.environ['CUDA_VISIBLE_DEVICES'] = GPU_IDS
WORK = Path('/kaggle/working/seal-ocr-rebuild')
PADDLE = Path('/kaggle/working/PaddleOCR')
RUNS = Path('/kaggle/working/runs')
for folder in (WORK, RUNS):
    if folder.exists(): shutil.rmtree(folder)
    folder.mkdir(parents=True)
print({'mode': RUN_MODE, 'controlEpochs': CONTROL_EPOCHS, 'fineTuneEpochs': FINETUNE_EPOCHS, 'gpus': GPU_IDS})


## Locate and verify the complete v4 dataset

The input may be an extracted folder or ZIP file. Every image hash and the dataset fingerprint are checked before training.


In [ ]:
WANTED = {'seal-recognition-v4-manual'}
found = {version: [] for version in WANTED}

def register_metadata(path):
    try:
        metadata = json.loads(path.read_text())
    except Exception:
        return
    version = metadata.get('datasetVersion')
    if version in found and (path.parent / 'manifest.csv').is_file():
        found[version].append((path.parent, metadata))

for metadata_path in Path('/kaggle/input').rglob('metadata.json'):
    register_metadata(metadata_path)

missing = {version for version, matches in found.items() if not matches}
for index, zip_path in enumerate(Path('/kaggle/input').rglob('*.zip')):
    if not missing: break
    with ZipFile(zip_path) as archive:
        names = archive.namelist()
        if any(Path(name).is_absolute() or '..' in Path(name).parts for name in names):
            raise ValueError(f'Unsafe ZIP path: {zip_path}')
        versions = set()
        for name in names:
            if Path(name).name != 'metadata.json': continue
            try: versions.add(json.loads(archive.read(name))['datasetVersion'])
            except Exception: pass
        if not (versions & missing): continue
        destination = WORK / 'datasets' / f'zip-{index}'
        archive.extractall(destination)
        for metadata_path in destination.rglob('metadata.json'):
            register_metadata(metadata_path)
        missing = {version for version, matches in found.items() if not matches}

for version, matches in found.items():
    if len(matches) != 1: raise RuntimeError(f'Expected exactly one {version} dataset, found {len(matches)}')

def verify_dataset(root, metadata):
    total = metadata.get('totalSamples', metadata.get('totalVerified'))
    train_total = metadata['trainSamples']
    val_total = metadata.get('validationSamples', metadata.get('valSamples'))
    if total != train_total + val_total: raise RuntimeError(f'Bad counts: {metadata}')
    rows = list(csv.DictReader((root / 'manifest.csv').open(encoding='utf-8-sig')))
    if len(rows) != total: raise RuntimeError(f'Manifest count {len(rows)} != {total}')
    fingerprint_lines = []
    for row in rows:
        image = root / row['image']
        digest = hashlib.sha256(image.read_bytes()).hexdigest()
        if row.get('sha256') and digest != row['sha256']: raise RuntimeError(f'Hash mismatch: {image}')
        fingerprint_lines.append(f"{row['image']}\t{row['ground_truth']}\t{digest}")
    fingerprint = hashlib.sha256('\n'.join(sorted(fingerprint_lines)).encode()).hexdigest()
    if fingerprint != metadata.get('datasetSha256'): raise RuntimeError(f'Dataset fingerprint mismatch: {root}')
    return rows

V4, v4_metadata = found['seal-recognition-v4-manual'][0]
v4_rows = verify_dataset(V4, v4_metadata)
base_train_lines = [f"{row['image']}\t{row['ground_truth']}" for row in v4_rows if row['split'] == 'train' and row.get('source_dataset') == 'seal-recognition-v3']
base_val_lines = [f"{row['image']}\t{row['ground_truth']}" for row in v4_rows if row['split'] == 'val' and row.get('source_dataset') == 'seal-recognition-v3']
if (len(base_train_lines), len(base_val_lines)) != (13000, 1500): raise RuntimeError('The v4 dataset does not contain the complete frozen v3 split')
CONTROL_TRAIN_LABEL = WORK / 'v3_train.txt'
CONTROL_VAL_LABEL = WORK / 'v3_val.txt'
CONTROL_TRAIN_LABEL.write_text('\n'.join(base_train_lines) + '\n')
CONTROL_VAL_LABEL.write_text('\n'.join(base_val_lines) + '\n')
print({'dataset': str(V4), 'v4Rows': len(v4_rows), 'derivedV3Train': len(base_train_lines), 'derivedV3Val': len(base_val_lines)})


In [ ]:
# Dependency flow preserved from the working v3 notebook.
if not PADDLE.exists():
    subprocess.run(['git', '-c', 'advice.detachedHead=false', 'clone', '--depth', '1', '--branch', 'v3.7.0', 'https://github.com/PaddlePaddle/PaddleOCR.git', str(PADDLE)], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '-r', str(PADDLE / 'requirements.txt'), 'pyyaml'], check=True)
try:
    import paddle
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-q', '--upgrade', 'paddlepaddle-gpu==3.3.0', '-i', 'https://www.paddlepaddle.org.cn/packages/stable/cu126/'], check=True)
    import paddle
if not paddle.device.is_compiled_with_cuda(): raise RuntimeError('Paddle has no CUDA support. Start a fresh Kaggle GPU session.')
if paddle.device.cuda.device_count() < 2: raise RuntimeError('This notebook requires a Kaggle 2-GPU accelerator.')
paddle.utils.run_check()
print({'paddle': paddle.__version__, 'visibleGpuCount': paddle.device.cuda.device_count(), 'device': paddle.device.get_device()})


In [ ]:
# Same train-only spacing transform used by production v1.
transform_source = r'''import random
import cv2
from .text_image_aug import tia_stretch

class RandomCharacterSpacing:
    def __init__(self, prob=0.35, min_segments=3, max_segments=6, **kwargs):
        self.prob=float(prob); self.min_segments=int(min_segments); self.max_segments=int(max_segments)
    def __call__(self, data):
        image=data["image"]
        if random.random() > self.prob or image.shape[0] < 20 or image.shape[1] < 20: return data
        height=image.shape[0]
        stretched=tia_stretch(image, random.randint(self.min_segments, self.max_segments))
        if stretched.shape[0] != height:
            stretched=cv2.resize(stretched, (stretched.shape[1], height), interpolation=cv2.INTER_CUBIC)
        data["image"]=stretched
        return data
'''
target = PADDLE / 'ppocr/data/imaug/random_character_spacing.py'
target.write_text(transform_source)
init_path = PADDLE / 'ppocr/data/imaug/__init__.py'
import_line = 'from .random_character_spacing import RandomCharacterSpacing\n'
if import_line not in init_path.read_text(): init_path.write_text(init_path.read_text() + '\n' + import_line)
print({'transformSha256': hashlib.sha256(target.read_bytes()).hexdigest()})


In [ ]:
import yaml
OFFICIAL_CONFIG = WORK / 'PP-OCRv6_medium_rec.yml'
urllib.request.urlretrieve('https://raw.githubusercontent.com/PaddlePaddle/PaddleOCR/v3.7.0/configs/rec/PP-OCRv6/PP-OCRv6_medium_rec.yml', OFFICIAL_CONFIG)
OFFICIAL_PRETRAIN = WORK / 'PP-OCRv6_medium_rec_pretrained.pdparams'
urllib.request.urlretrieve('https://paddle-model-ecology.bj.bcebos.com/paddlex/official_pretrained_model/PP-OCRv6_medium_rec_pretrained.pdparams', OFFICIAL_PRETRAIN)

def build_config(run_dir, dataset, train_label, val_label, epochs, seed, learning_rate):
    config = yaml.safe_load(OFFICIAL_CONFIG.read_text())
    config['Global'].update({'epoch_num': epochs, 'seed': seed, 'save_model_dir': str(run_dir / 'checkpoints'), 'save_epoch_step': 10, 'eval_batch_step': [0, 100], 'distributed': True, 'use_visualdl': False, 'max_text_length': 25})
    config['Optimizer']['lr'].update({'learning_rate': learning_rate, 'warmup_epoch': 1 if epochs < 10 else 2})
    transforms = [
        {'DecodeImage': {'img_mode': 'BGR', 'channel_first': False}},
        {'RecAug': {'tia_prob': 0.10, 'crop_prob': 0.0, 'reverse_prob': 0.0, 'noise_prob': 0.15, 'jitter_prob': 0.05, 'blur_prob': 0.10, 'hsv_aug_prob': 0.15}},
        {'RandomCharacterSpacing': {'prob': SPACING_PROB, 'min_segments': 3, 'max_segments': 6}},
        {'MultiLabelEncode': {'gtc_encode': 'NRTRLabelEncode'}},
        {'KeepKeys': {'keep_keys': ['image', 'label_ctc', 'label_gtc', 'length', 'valid_ratio']}},
    ]
    config['Train']['dataset'].update({'data_dir': str(dataset), 'label_file_list': [str(train_label)], 'transforms': transforms, 'ext_op_transform_idx': 1, 'ratio_list': [1.0]})
    config['Eval']['dataset'].update({'data_dir': str(dataset), 'label_file_list': [str(val_label)]})
    config['Train']['sampler']['first_bs'] = 32  # 32 per GPU x 2 = old global batch 64
    config['Train']['loader'].update({'batch_size_per_card': 32, 'drop_last': False, 'num_workers': 2})
    config['Eval']['loader'].update({'batch_size_per_card': 32, 'num_workers': 2})
    path = run_dir / 'config.yml'
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return path

def train_and_export(config, run_dir, pretrained):
    overrides = [f'Global.pretrained_model={pretrained}', f'Global.save_model_dir={run_dir / "checkpoints"}']
    command = [sys.executable, '-m', 'paddle.distributed.launch', '--gpus', GPU_IDS, '--log_dir', str(run_dir / 'distributed_logs'), 'tools/train.py', '-c', str(config), '-o', *overrides]
    subprocess.run(command, cwd=PADDLE, check=True)
    best = run_dir / 'checkpoints' / 'best_accuracy'
    if not best.with_suffix('.pdparams').is_file(): raise RuntimeError(f'Missing checkpoint: {best}.pdparams')
    inference = run_dir / 'inference'
    subprocess.run([sys.executable, 'tools/export_model.py', '-c', str(config), '-o', 'Global.distributed=False', f'Global.pretrained_model={best}', f'Global.save_inference_dir={inference}'], cwd=PADDLE, check=True)
    return best, inference

def package_run(run_dir, package_name):
    package_dir = WORK / 'packages' / package_name
    if package_dir.exists(): shutil.rmtree(package_dir)
    (package_dir / 'checkpoints').mkdir(parents=True)
    shutil.copy2(run_dir / 'config.yml', package_dir / 'config.yml')
    shutil.copy2(run_dir / 'run_metadata.json', package_dir / 'run_metadata.json')
    shutil.copytree(run_dir / 'inference', package_dir / 'inference')
    for suffix in ('pdparams', 'pdopt', 'states'):
        source = run_dir / 'checkpoints' / f'best_accuracy.{suffix}'
        if not source.is_file(): raise RuntimeError(f'Missing resumable checkpoint file: {source}')
        shutil.copy2(source, package_dir / 'checkpoints' / source.name)
    return shutil.make_archive(f'/kaggle/working/{package_name}', 'zip', package_dir)


## Stage 1 — rebuild the v1 checkpoint


In [ ]:
CONTROL_DIR = RUNS / 'seal-ocr-rec-v1-rebuilt'
CONTROL_DIR.mkdir(parents=True)
CONTROL_CONFIG = build_config(CONTROL_DIR, V4, CONTROL_TRAIN_LABEL, CONTROL_VAL_LABEL, CONTROL_EPOCHS, CONTROL_SEED, 0.00005)
CONTROL_BEST, CONTROL_INFERENCE = train_and_export(CONTROL_CONFIG, CONTROL_DIR, OFFICIAL_PRETRAIN)
control_metadata = {'modelVersion': 'seal-ocr-rec-v1-rebuilt', 'stage': 'control', 'datasetVersion': 'seal-recognition-v3-derived-from-v4', 'parentDatasetSha256': v4_metadata['datasetSha256'], 'trainSamples': len(base_train_lines), 'valSamples': len(base_val_lines), 'seed': CONTROL_SEED, 'epochs': CONTROL_EPOCHS, 'learningRate': 0.00005, 'spacingProbability': SPACING_PROB, 'gpus': GPU_IDS, 'globalBaseBatch': 64}
(CONTROL_DIR / 'run_metadata.json').write_text(json.dumps(control_metadata, indent=2))
CONTROL_ARCHIVE = package_run(CONTROL_DIR, 'seal-ocr-rec-v1-rebuilt')
print({'bestCheckpoint': str(CONTROL_BEST), 'inference': str(CONTROL_INFERENCE), 'archive': CONTROL_ARCHIVE})


## Stage 2 — low-learning-rate fine-tune on v4


In [ ]:
base_lines = base_train_lines
manual_lines = [f"{row['image']}\t{row['ground_truth']}" for row in v4_rows if row['split'] == 'train' and row.get('source_dataset') == 'manual-user-100']
weighted_lines = base_lines + manual_lines * MANUAL_REPEAT
random.Random(FINETUNE_SEED).shuffle(weighted_lines)
FINETUNE_LABEL = WORK / 'v4_train_weighted.txt'
FINETUNE_LABEL.write_text('\n'.join(weighted_lines) + '\n')
print({'baseRows': len(base_lines), 'manualRows': len(manual_lines), 'manualRepeat': MANUAL_REPEAT, 'effectiveTrainRows': len(weighted_lines)})

FINETUNE_DIR = RUNS / 'seal-ocr-rec-v5-candidate'
FINETUNE_DIR.mkdir(parents=True)
FINETUNE_CONFIG = build_config(FINETUNE_DIR, V4, FINETUNE_LABEL, V4 / 'val.txt', FINETUNE_EPOCHS, FINETUNE_SEED, 0.00001)
FINETUNE_BEST, FINETUNE_INFERENCE = train_and_export(FINETUNE_CONFIG, FINETUNE_DIR, CONTROL_BEST)
finetune_metadata = {'modelVersion': 'seal-ocr-rec-v5-candidate', 'stage': 'fine-tune', 'datasetVersion': v4_metadata['datasetVersion'], 'datasetSha256': v4_metadata['datasetSha256'], 'parentCheckpoint': 'seal-ocr-rec-v1-rebuilt/best_accuracy', 'seed': FINETUNE_SEED, 'epochs': FINETUNE_EPOCHS, 'learningRate': 0.00001, 'spacingProbability': SPACING_PROB, 'manualRepeat': MANUAL_REPEAT, 'gpus': GPU_IDS, 'globalBaseBatch': 64}
(FINETUNE_DIR / 'run_metadata.json').write_text(json.dumps(finetune_metadata, indent=2))
FINETUNE_ARCHIVE = package_run(FINETUNE_DIR, 'seal-ocr-rec-v5-candidate')
print({'bestCheckpoint': str(FINETUNE_BEST), 'inference': str(FINETUNE_INFERENCE), 'archive': FINETUNE_ARCHIVE})
print('Download both ZIP files. Benchmark both inference folders; promote v5 only at >=270/300.')
